In [2]:
#Load Dataset

import pandas as pd

# Load the dataset
df = pd.read_csv("SuperMarket Analysis.csv")

# Display first 5 rows
df.head()

,Invoice ID,Branch,City,Customer type,Gender,Product line,Unit price,Quantity,Tax 5%,Sales,Date,Time,Payment,cogs,gross margin percentage,gross income,Rating
0,750-67-8428,Alex,Yangon,Member,Female,Health and beauty,74.69,7,26.1415,548.9715,1/5/2019,1:08:00 PM,Ewallet,522.83,4.761905,26.1415,9.1
1,226-31-3081,Giza,Naypyitaw,Normal,Female,Electronic accessories,15.28,5,3.8200,80.2200,3/8/2019,10:29:00 AM,Cash,76.40,4.761905,3.8200,9.6
2,631-41-3108,Alex,Yangon,Normal,Female,Home and lifestyle,46.33,7,16.2155,340.5255,3/3/2019,1:23:00 PM,Credit card,324.31,4.761905,16.2155,7.4
3,123-19-1176,Alex,Yangon,Member,Female,Health and beauty,58.22,8,23.2880,489.0480,1/27/2019,8:33:00 PM,Ewallet,465.76,4.761905,23.2880,8.4
4,373-73-7910,Alex,Yangon,Member,Female,Sports and travel,86.31,7,30.2085,634.3785,2/8/2019,10:37:00 AM,Ewallet,604.17,4.761905,30.2085,5.3


In [3]:
#Check Missing Values and Duplicate Rows

# Check missing values
print("Missing Values:")
print(df.isnull().sum())

# Check duplicate rows
print("\nDuplicate Rows:")
print(df.duplicated().sum())

Missing Values:
Invoice ID                 0
Branch                     0
City                       0
Customer type              0
Gender                     0
Product line               0
Unit price                 0
Quantity                   0
Tax 5%                     0
Sales                      0
Date                       0
Time                       0
Payment                    0
cogs                       0
gross margin percentage    0
gross income               0
Rating                     0
dtype: int64

Duplicate Rows:
0


In [4]:
#Clean Dataset

df = df.dropna()

df = df.drop_duplicates()

df = df.reset_index(drop=True)

print("Shape of Cleaned Dataset:", df.shape)

Shape of Cleaned Dataset: (1000, 17)


In [5]:
#Remove Unnecessary Columns

df = df.drop(["Invoice ID", "Date", "Time"], axis=1)

print(df.columns)

Index(['Branch', 'City', 'Customer type', 'Gender', 'Product line',
       'Unit price', 'Quantity', 'Tax 5%', 'Sales', 'Payment', 'cogs',
       'gross margin percentage', 'gross income', 'Rating'],
      dtype='object')


In [6]:
#Create Demand Score

df["Demand Score"] = (
    df["Quantity"] * 0.5
    + df["Rating"] * 0.3
    + (df["Unit price"] < df["Unit price"].median()).astype(int) * 0.2
)

print(df[["Quantity", "Rating", "Unit price", "Demand Score"]].head())

   Quantity  Rating  Unit price  Demand Score
0         7     9.1       74.69          6.23
1         5     9.6       15.28          5.58
2         7     7.4       46.33          5.92
3         8     8.4       58.22          6.52
4         7     5.3       86.31          5.09


In [7]:
#Create Demand Feature

median_score = df["Demand Score"].median()

df["Demand"] = df["Demand Score"].apply(
    lambda x: "High Demand" if x >= median_score else "Low Demand"
)

print(df["Demand"].value_counts())

Demand
High Demand    500
Low Demand     500
Name: count, dtype: int64


In [8]:
#Separate Features and Target

# Remove only the target, quantity and unnecessary identification columns
X = df.drop(
    [
        "Demand",
        "Quantity"
    ],
    axis=1,
    errors="ignore"
)

# Target Feature
y = df["Demand"]

print("Input Features:")
print(X.head())

print("\nTarget Feature:")
print(y.head())

print("\nFeatures Used:")
print(X.columns)

Input Features:
  Branch       City Customer type  Gender            Product line  Unit price  \
0   Alex     Yangon        Member  Female       Health and beauty       74.69   
1   Giza  Naypyitaw        Normal  Female  Electronic accessories       15.28   
2   Alex     Yangon        Normal  Female      Home and lifestyle       46.33   
3   Alex     Yangon        Member  Female       Health and beauty       58.22   
4   Alex     Yangon        Member  Female       Sports and travel       86.31   

    Tax 5%     Sales      Payment    cogs  gross margin percentage  \
0  26.1415  548.9715      Ewallet  522.83                 4.761905   
1   3.8200   80.2200         Cash   76.40                 4.761905   
2  16.2155  340.5255  Credit card  324.31                 4.761905   
3  23.2880  489.0480      Ewallet  465.76                 4.761905   
4  30.2085  634.3785      Ewallet  604.17                 4.761905   

   gross income  Rating  Demand Score  
0       26.1415     9.1          6.2

In [16]:
print(df[["Demand Score", "Demand"]].head())

   Demand Score       Demand
0          6.23  High Demand
1          5.58  High Demand
2          5.92  High Demand
3          6.52  High Demand
4          5.09  High Demand


In [9]:
#Convert Categorical Features into Numeric Values

# Convert categorical columns into numeric values
X = pd.get_dummies(X, drop_first=True)

# Display converted features
print(X.head())

   Unit price   Tax 5%     Sales    cogs  gross margin percentage  \
0       74.69  26.1415  548.9715  522.83                 4.761905   
1       15.28   3.8200   80.2200   76.40                 4.761905   
2       46.33  16.2155  340.5255  324.31                 4.761905   
3       58.22  23.2880  489.0480  465.76                 4.761905   
4       86.31  30.2085  634.3785  604.17                 4.761905   

   gross income  Rating  Demand Score  Branch_Cairo  Branch_Giza  ...  \
0       26.1415     9.1          6.23         False        False  ...   
1        3.8200     9.6          5.58         False         True  ...   
2       16.2155     7.4          5.92         False        False  ...   
3       23.2880     8.4          6.52         False        False  ...   
4       30.2085     5.3          5.09         False        False  ...   

   City_Yangon  Customer type_Normal  Gender_Male  \
0         True                 False        False   
1        False                  True    

In [10]:
X.shape

(1000, 21)

In [11]:
#Convert Target Feature into Numeric Values

from sklearn.preprocessing import LabelEncoder

# Create LabelEncoder object
le = LabelEncoder()

# Convert Demand into numeric values
y = le.fit_transform(y)

# Display converted target values
print(y)

[0 0 0 0 0 1 1 0 1 1 1 1 1 0 0 1 1 0 1 1 1 1 1 0 1 0 1 1 0 0 1 0 0 1 1 1 1
 0 0 0 1 1 1 0 1 1 0 0 0 0 0 0 1 1 1 0 1 0 0 0 1 0 0 0 1 0 1 0 0 1 0 0 1 0
 0 0 0 0 0 1 1 1 0 1 1 1 0 0 1 1 1 0 0 0 1 1 0 1 1 0 1 0 0 0 0 0 0 0 0 0 1
 1 0 0 0 1 1 1 0 1 0 0 0 0 0 1 0 0 0 0 0 0 1 0 1 1 0 0 0 0 0 0 0 1 0 0 0 1
 0 0 1 0 0 0 0 0 1 0 0 0 0 1 0 1 0 0 0 0 0 0 1 0 0 0 0 0 0 0 1 1 0 0 0 0 0
 1 0 1 1 1 1 1 0 1 0 1 1 1 1 1 0 1 0 0 1 0 0 0 1 0 0 0 1 1 0 1 0 1 0 1 0 0
 0 1 1 1 0 0 1 0 1 1 0 1 0 1 1 0 1 1 0 1 1 1 0 0 1 1 1 1 0 0 1 1 0 1 1 0 1
 1 0 1 1 0 1 0 1 0 1 0 1 1 0 1 0 0 0 1 0 0 0 1 1 0 0 1 1 0 0 1 0 0 1 1 0 0
 1 1 1 1 1 1 1 1 0 1 0 0 1 1 1 1 1 0 0 1 1 1 0 1 0 1 0 1 0 1 1 1 1 1 0 1 1
 1 0 0 0 1 1 0 1 0 0 0 1 0 1 0 0 0 0 0 0 0 1 1 0 0 1 1 0 0 1 0 1 1 0 1 0 0
 0 1 1 0 0 1 0 0 1 1 1 1 0 1 0 1 0 0 0 0 1 0 1 0 1 1 1 0 0 0 0 1 1 0 0 1 0
 1 1 1 0 1 0 1 0 1 1 1 1 0 1 1 0 0 1 0 0 1 1 0 1 1 0 0 1 0 1 0 1 0 0 0 0 1
 0 1 0 1 1 1 0 0 1 1 1 1 0 0 0 1 1 0 1 1 0 0 0 1 1 1 1 0 0 0 1 1 1 1 0 1 0
 0 1 0 0 1 0 1 1 0 1 0 0 

In [12]:
#Train-Test Split

from sklearn.model_selection import train_test_split

# Split the dataset
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

# Display shapes
print("Training Data:", X_train.shape)
print("Testing Data:", X_test.shape)

Training Data: (800, 21)
Testing Data: (200, 21)


In [13]:
#Feature Scaling

from sklearn.preprocessing import StandardScaler

# Create StandardScaler object
scaler = StandardScaler()

# Scale training data
X_train_scaled = scaler.fit_transform(X_train)

# Scale testing data
X_test_scaled = scaler.transform(X_test)

In [14]:
#Import Evaluation Metrics

from sklearn.metrics import accuracy_score
from sklearn.metrics import precision_score
from sklearn.metrics import recall_score
from sklearn.metrics import f1_score

In [15]:
#SVM with Linear Kernel

from sklearn.svm import SVC

# Create Linear SVM model
svm_linear = SVC(kernel="linear")

# Train the model
svm_linear.fit(X_train_scaled, y_train)

# Make predictions
y_pred_linear = svm_linear.predict(X_test_scaled)

# Calculate performance metrics
accuracy_linear = accuracy_score(y_test, y_pred_linear)
precision_linear = precision_score(y_test, y_pred_linear)
recall_linear = recall_score(y_test, y_pred_linear)
f1_linear = f1_score(y_test, y_pred_linear)

# Display results
print("SVM Linear Results")
print("Accuracy:", accuracy_linear)
print("Precision:", precision_linear)
print("Recall:", recall_linear)
print("F1 Score:", f1_linear)

SVM Linear Results
Accuracy: 0.985
Precision: 0.9797979797979798
Recall: 0.9897959183673469
F1 Score: 0.9847715736040609
